In [ ]:
import json
import glob
import re
import pandas as pd
from pathlib import Path
from natsort import natsorted
from format_tex import format_result
import os

In [ ]:
RUN_LABEL = 'news999'
DATASETS = os.listdir(Path('../results/') / RUN_LABEL)
DATASET_PATH = Path('../data/classification/news_ideology')
DATASET_INDEX = 'Unnamed: 0'
RESULT_DIRS = [
    Path('../results/') / RUN_LABEL / dataset for dataset in DATASETS
]
LABELS = ['Liberal', 'Neutral', 'Conservative']

In [ ]:
results = []
for result_dir in RESULT_DIRS:
    results += glob.glob(str(result_dir / '**/balanced_bertscore/8000_cands*/**/test-1000.json'), recursive=True)
    results += glob.glob(str(result_dir /  '**/random/8000_cands*/**/test-1000.json'), recursive=True)
results = [i for i in results if 'GPT' in i]
results

In [ ]:
def sanitize_prediction(x):
    if 'neutral' in x.lower():
        return 'Neutral'
    if 'liberal' in x.lower():
        return 'Liberal'
    if 'conservative' in x.lower():
        return 'Conservative'
    raise Exception('Invalid response', x)

In [ ]:
llm_names = set()
num_shots_ = set()
datasets = set()

test_set = pd.read_json(DATASET_PATH / 'test_small.json')
test_set = test_set.set_index('article_id')
test_set['true'] = test_set['label'].map(lambda x : LABELS[x])

keys = []

for result in results:
    regex = f'../results/{RUN_LABEL}/(?P<dataset>.*?)/test/(?P<num_shots>[0-9]+)?_shots/.*?/[0-9]+?_cands.*?/s0/(?P<llm_name>.*)?/test-1000.json'
    groups = re.search(
        re.compile(regex),
        result
    )

    llm_name = groups.group('llm_name')
    num_shots = groups.group('num_shots')
    dataset = groups.group('dataset')

    llm_names.add(llm_name)
    num_shots_.add(num_shots)
    datasets.add(dataset)

    print(llm_name, num_shots, dataset)

    with open(result) as f:
        js = json.load(f)

    key = f'{llm_name}_{num_shots}_{dataset}'
    keys.append(key)
    
    rows = []
    for res in js['results']:
        try:
            pred = sanitize_prediction(res['pred'])
            idx = res['article_id']
            rows.append({
                key: pred,
                'idx': idx
            })
        except Exception as e:
            print(e)
            continue
    df = pd.DataFrame(rows).set_index('idx')
    test_set = test_set.merge(df, left_index=True, right_index=True)

keys = natsorted(keys)
test_set = test_set.dropna(subset=keys)

In [ ]:
for dataset in sorted(datasets, key=len):
    print('\\multicolumn{9}{c}{%s}\\\\ \\midrule' % dataset)
    for llm_name in sorted(llm_names):
        for num_shots in natsorted(num_shots_):
            key = f'{llm_name}_{num_shots}_{dataset}'
            if key in test_set.columns:
                print(format_result(llm_name, num_shots, test_set['true'], test_set[key], LABELS), end='\\\\\n')

In [ ]:
last_model = None
batch = []
for dataset in sorted(datasets, key=len):
    for model in sorted(llm_names):
        for num_shots in sorted(num_shots_):
            if last_model and model != last_model:
                print('\\multirow{%s}{*}{%s}\n' % (len(batch), last_model), end='')
                print(' \\\\ \n'.join(batch), end='')
                print(' \\\\ \n \\midrule')
                batch = []
            batch.append(format_result('', num_shots, test_set['true'], test_set[key], labels=LABELS))
            last_model = model

if len(batch) > 0:
    print('\\multirow{%s}{*}{%s}\n' % (len(batch), last_model), end='')
    print(' \\\\ \n'.join(batch), end='')
    print(' \\\\ \n')
    batch = []